In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans

In [ ]:
class KMeans_HM:
    def __init__(self, n_clusters, max_iter=100):
        self.n_clusters = n_clusters
        self.max_iter = max_iter
        self.centroids = None
        self.labels = None
    

    def fit_predict (self, X, gradual_plot = True):
        """Run the K-means clustering on X.
        : param X: input data points, array, shape = (N,C). (5000, 2)
        : return: labels : array, shape = N.
        """

        # ignore, just for the gradual plot
        if gradual_plot:
            plt.figure(figsize=(15,15))
            plot_idx = 1

        # 1. initialize K random centroids in the data space (only once)
        initial_centroids = self.initilization_centroids(X)
    
        for idx in range(self.max_iter):
            # 2. compute distance matrix
            distance_matrix = self.d_from_to_matrix(X, initial_centroids)

            # 3. use the 2D from-to matrix to get closest centroid for each point
            labels = self.get_clusters(distance_matrix)

            # 4. recompute centroids
            centroids = self.get_new_centroids(X, distance_matrix, labels)

            # 5. update the new centroids
            initial_centroids = centroids

            # ignore, just for the gradual plot
            if gradual_plot and idx%10==0:
                plt.subplot(5,3, plot_idx)
                sns.scatterplot(x=X[:,0], y=X[:,1], hue=labels, legend=False)
                sns.scatterplot(x=centroids[:,0], y=centroids[:,1], s=80, marker='X')
                plt.title(f"Iteration {idx}")

                plot_idx += 1
        
        # ignore, just for the gradual plot
        if gradual_plot:
            plt.tight_layout()
            plt.show()

        self.labels = labels
        self.centroids = centroids
        return labels, centroids
    






    def initilization_centroids(self, X):
        # get random rows from data as initial centroids
        initial_centroids_idx = np.random.randint(low = 0, high=len(X), size=self.n_clusters)       # take these rows
        initial_centroids = X[initial_centroids_idx]

        return initial_centroids


    def d_from_to_matrix(self, X, initial_centroids):
        # expand dimension to allow broadcasting:
            # build from-to matrix, distance from a point (row) to a centroid (column)
            # (5000, 2)
            # (15, 2)
            # I want (5000, 15), then:
            # (5000, 2) --> (5000,  *1*,  2)
            # (15, 2)   --> (*1*,     15, 2)
        X_expanded = X.reshape(X.shape[0], 1, 2)
        initial_centroids_expanded = initial_centroids.reshape(1, initial_centroids.shape[0], 2)

        # compute euclidean distance + save it in a 2D from-to matrix
        difference = X_expanded - initial_centroids_expanded
        sq_difference = difference**2
        sum_sq_difference = np.sum(sq_difference, axis=2)
        distance_matrix = sum_sq_difference**0.5
        
        return distance_matrix
    
    def get_clusters(self, distance_matrix):
        labels = []
        for row in distance_matrix:
            # pick smallest values from row, USE ARGMIN TO PICK THE INDEX SO THAT YOU ALSO KNOW THE CLUSTER
            row_cluster = row.argmin()
            labels.append(row_cluster)

        return labels

    def get_new_centroids(self, X, distance_matrix, cluster_labels):
        # the new centroid is the **mean** **of the coordinates** of all points inside that cluster
        
        # to make things easier use a df to associate each points to its cluster
        distance_matrix_df = pd.DataFrame(distance_matrix)
        distance_matrix_df['cluster'] = cluster_labels

        # exploit the groupby to group the points that belongs to the same cluster together and use their index to get their coordinates
        centroids = []
        grouped = distance_matrix_df.groupby('cluster')
        for id, values in grouped:
            idx_points_same_cluster = values.index
            # get the coordinates of points with same values, take the X values and do the mean, same on the Y values = new_centroid
            centroid = [X[idx_points_same_cluster][:,0].mean(), X[idx_points_same_cluster][:,1].mean()]
            centroids.append(centroid)
        
        # convert the list into an array to make things quicker
        centroids = np.array(centroids)
        
        return centroids



# '''--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------'''

def final_plot(X, labels, centroids):
    plt.figure(figsize=(8,6))
    sns.scatterplot(x=X[:,0], y=X[:,1], hue=labels, legend=False)
    sns.scatterplot(x=centroids[:,0], y=centroids[:,1], s=80, marker='X')
    plt.title('Final clustering')
    plt.show()


if __name__ == '__main__':
    df = pd.read_csv('../../Dataset/LAB8/2D_gauss_clusters.txt', sep = ',')
    X = df.values
    
    kmeans = KMeans_HM(15)
    labels, centroids = kmeans.fit_predict(X, gradual_plot=True)

    final_plot(X, labels, centroids)

#### Let's try to compare it with KMeans actual clustering

In [ ]:
kmeans = KMeans(n_clusters=14, max_iter=100, random_state=42)
kmeans.fit_predict(X)

centroids = kmeans.cluster_centers_
clusters = kmeans.labels_
SSE = kmeans.inertia_

plt.figure(figsize=(8,6))
sns.scatterplot(x=X[:,0], y=X[:,1], hue=clusters, legend=False)
sns.scatterplot(x=centroids[:,0], y=centroids[:,1], s=80, marker='X')
plt.title('Clustering actual OG KMeans')
plt.show()

#### The results are very similar, I believe I did a decent job :) (in 3 fucking days)

---

#### 4. Once you get to a fully functional version of your class, load also the Chameleon data (see Section 2.2) and run the K-means algorithm on both the datasets. Feel free to run multiple times the algorithm varying n_clusters

In [ ]:
df = pd.read_csv('../../Dataset/LAB8/chameleon_clusters.txt', sep=',')
display(df)
print()
# no way .values actually returns an array from the DF
X = df.values
print(df.values)
print(df.values.shape)
print(type(df.values))

In [ ]:
# first let's inspect the data distribution
plt.figure(figsize=(8,6))
sns.scatterplot(x=X[:,0], y=X[:,1])
plt.title('Chamaleon data distribution')
plt.show()


#### We can already say the KMeans will perform poorly:
- no globular clusters
- weird shapes
- very dense data space
- there's noise and outliers

Let's still see the performance, there are 6 clusters

In [ ]:
# let's see how the home-made KMeans performs
kmeans = KMeans_HM(n_clusters=6)
labels, centroids = kmeans.fit_predict(X)
# should be just kmeans.fit_predict(X) and then you should recall like kmeans.labels

plt.figure(figsize=(8,6))
sns.scatterplot(x=X[:,0], y=X[:,1], hue=labels, legend=False)
sns.scatterplot(x=centroids[:,0], y=centroids[:,1], s=80, marker='X')
plt.title('KMeans clustering')
plt.show()


#### That's bad, let's try to increase the number of clusters

In [ ]:
kmeans = KMeans_HM(n_clusters=10)
labels, centroids = kmeans.fit_predict(X, gradual_plot=False)
# should be just kmeans.fit_predict(X) and then you should recall like kmeans.labels

plt.figure(figsize=(8,6))
sns.scatterplot(x=X[:,0], y=X[:,1], hue=labels, legend=False)
sns.scatterplot(x=centroids[:,0], y=centroids[:,1], s=80, marker='X')
plt.title('KMeans clustering')
plt.show()

#### Nah, we need a different clustering approach